In [1]:
import pandas as pd
import numpy as np
import os
os.environ["CUDA_VISIBLE_DEVICES"]="1"
import torch
from transformers import AutoTokenizer, AutoModel, AutoConfig,  BitsAndBytesConfig


/home/dataconv/anaconda3/envs/sf_rag_djk/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_storage=torch.bfloat16,
)

# load model with tokenizer
model = AutoModel.from_pretrained(
    'nvidia/NV-Embed-v2', 
    trust_remote_code=True,
    quantization_config = bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
model.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:05<00:00,  1.30s/it]


NVEmbedModel(
  (latent_attention_model): LatentAttentionModel(
    (cross_attend_blocks): ModuleList(
      (0): PreNorm(
        (fn): Attention(
          (to_q): Linear4bit(in_features=4096, out_features=32768, bias=False)
          (to_kv): Linear4bit(in_features=4096, out_features=65536, bias=False)
          (to_out): Linear4bit(in_features=32768, out_features=4096, bias=False)
        )
        (norm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
        (norm_context): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
      )
      (1): PreNorm(
        (fn): FeedForward(
          (net): Sequential(
            (0): Linear4bit(in_features=4096, out_features=32768, bias=True)
            (1): GEGLU()
            (2): Linear4bit(in_features=16384, out_features=4096, bias=True)
          )
        )
        (norm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
      )
    )
  )
  (embedding_model): BidirectionalMistralModel(
    (embed_tokens): Embedding(

In [5]:
from tqdm import tqdm
from datasets import Dataset
from torch.utils.data import DataLoader

batchsize = 8
max_len = 1024

data_folder = '/raid/deallab/SF_RAG_Data/ASQA'
# data_folder = '../data'

# read evidence data
evidence_test_path = f'{data_folder}/test/evidence_test_eval.csv'
evidence_df = pd.read_csv(evidence_test_path)
evidence_test_title_path = f'{data_folder}/test/evidence_test_eval_title.csv'

#dump directory
embedd_test_path = f'{data_folder}/test/embedd_test_eval_title.npy'
evidence_embeddings = []

#prepare data
evidence_title=evidence_df.drop_duplicates(['title'])
evidence_ds = Dataset.from_pandas(evidence_title)
evidence_title.to_csv(evidence_test_title_path,index=False)

In [10]:
len(evidence_ds)

1897

In [4]:
len(evidence_title)

1897

In [3]:
evidence_title

,sample_id,title,text
0,-7013890438520559398,International Federation of Football History &...,Document: International Federation of Football...
10,-7013890438520559398,List of FIFA World Cup records and statistics,Document: List of FIFA World Cup records and s...
42,-7013890438520559398,List of footballers with more than 50 internat...,Document: List of footballers with more than 5...
50,-7013890438520559398,List of women's footballers with 100 or more i...,Document: List of women's footballers with 100...
54,-7013890438520559398,List of footballers with 500 or more goals,Document: List of footballers with 500 or more...
...,...,...,...
21776,8651936974639934250,Singin' in the Rain (musical),Document: Singin' in the Rain (musical)\n\nSin...
21780,8651936974639934250,Singin' in the Rain (song),Document: Singin' in the Rain (song)\n\n\n\n\n...
21785,8651936974639934250,Jimmy Thompson (actor),# Jimmy Thompson (actor) #\nJames Edward Thomp...
21786,8651936974639934250,Singin' in the Rain,Document: Singin' in the Rain\n\nSingin' in th...


In [19]:
evidence_embeddings = []
evidence_dl = DataLoader(evidence_ds, batch_size=32, shuffle=False) #
for evidence in tqdm(evidence_dl):
    with torch.no_grad():
        evidence_embeddings.append(model.encode(evidence['title'], max_length = max_len).cpu().detach().numpy())
    
# turn to np arr
evidence_embeddings = np.concatenate(evidence_embeddings, axis=0)
print(evidence_embeddings.shape)
np.save(embedd_test_path, evidence_embeddings)

100%|██████████| 60/60 [00:16<00:00,  3.62it/s]

(1897, 4096)
